In [7]:
# ============================================================
# 🔧 COMPLETE BACKEND SETUP
# Run this ONE cell before launching the website
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    IsolationForest
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 1. CREATE ENERGY DATASET
# ============================================================

np.random.seed(42)

date_range = pd.date_range(
    start="2025-01-01",
    end="2025-12-31 23:00:00",
    freq="h"
)

df = pd.DataFrame({
    "DateTime": date_range
})

df["Date"] = df["DateTime"].dt.date
df["Hour"] = df["DateTime"].dt.hour
df["Day_of_Week"] = df["DateTime"].dt.dayofweek
df["Month"] = df["DateTime"].dt.month
df["Is_Weekend"] = (df["Day_of_Week"] >= 5).astype(int)

# Temperature
df["Temperature_C"] = (
    25
    + 7 * np.sin(2 * np.pi * (df["Month"] - 3) / 12)
    + 5 * np.sin(2 * np.pi * (df["Hour"] - 6) / 24)
    + np.random.normal(0, 1.5, len(df))
)

# Occupancy
df["Occupancy"] = (
    40
    + 90 * (
        (df["Hour"] >= 8) &
        (df["Hour"] <= 18)
    ).astype(int)
    + np.random.normal(0, 15, len(df))
)

df["Occupancy"] = df["Occupancy"].clip(0, 250)

# Peak hour
df["Peak_Hour"] = (
    ((df["Hour"] >= 9) & (df["Hour"] <= 12)) |
    ((df["Hour"] >= 17) & (df["Hour"] <= 20))
).astype(int)

# ============================================================
# 2. SIMULATE ENERGY CONSUMPTION
# ============================================================

base_load = 18

occupancy_load = df["Occupancy"] * 0.12

temperature_load = np.maximum(
    df["Temperature_C"] - 24, 0
) * 0.8

hour_load = (
    8 * np.sin(
        np.pi * (df["Hour"] - 6) / 12
    ).clip(0)
)

peak_load = df["Peak_Hour"] * 8

weekend_reduction = df["Is_Weekend"] * -5

noise = np.random.normal(0, 2, len(df))

df["Energy_Consumption_kWh"] = (
    base_load
    + occupancy_load
    + temperature_load
    + hour_load
    + peak_load
    + weekend_reduction
    + noise
)

df["Energy_Consumption_kWh"] = (
    df["Energy_Consumption_kWh"].clip(lower=5)
)

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

df["Previous_Hour_Consumption"] = (
    df["Energy_Consumption_kWh"].shift(1)
)

df["Previous_Day_Consumption"] = (
    df["Energy_Consumption_kWh"].shift(24)
)

df_ml = df.dropna().copy()

# ============================================================
# 4. FEATURES AND TARGET
# ============================================================

features = [
    "Hour",
    "Day_of_Week",
    "Month",
    "Is_Weekend",
    "Temperature_C",
    "Occupancy",
    "Peak_Hour",
    "Previous_Hour_Consumption",
    "Previous_Day_Consumption"
]

target = "Energy_Consumption_kWh"

X = df_ml[features]
y = df_ml[target]

# ============================================================
# 5. TIME-BASED TRAIN TEST SPLIT
# ============================================================

split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# ============================================================
# 6. TRAIN MODELS
# ============================================================

models = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

# ============================================================
# 7. MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE"
).reset_index(drop=True)

best_model_name = results_df.iloc[0]["Model"]

best_model = models[best_model_name]

# ============================================================
# 8. FEATURE IMPORTANCE
# ============================================================

rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# ============================================================
# 9. ANOMALY DETECTION
# ============================================================

anomaly_features = df[
    [
        "Energy_Consumption_kWh",
        "Temperature_C",
        "Occupancy",
        "Hour"
    ]
].copy()

isolation_model = IsolationForest(
    contamination=0.02,
    random_state=42
)

anomaly_prediction = isolation_model.fit_predict(
    anomaly_features
)

df["Anomaly"] = anomaly_prediction

df["Consumption_Status"] = np.where(
    df["Anomaly"] == -1,
    "Unusual",
    "Normal"
)

# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("=" * 65)
print("        SMART ENERGY PROJECT BACKEND READY")
print("=" * 65)

print(f"\nDataset Records : {len(df):,}")

print(f"\nBest Model      : {best_model_name}")

print("\nModel Performance:")
display(results_df)

print("\nTop 5 Important Features:")
display(feature_importance.head(5))

print("\nAnomaly Summary:")
print(df["Consumption_Status"].value_counts())

print("\n✅ Backend successfully created.")
print("✅ You can now run the website dashboard cell.")

        SMART ENERGY PROJECT BACKEND READY

Dataset Records : 8,760

Best Model      : Random Forest

Model Performance:


,Model,MAE,RMSE,R2
0,Random Forest,1.770626,2.251224,0.957637
1,Gradient Boosting,1.874906,2.380624,0.952627
2,Linear Regression,2.545972,3.147826,0.917173



Top 5 Important Features:


,Feature,Importance
0,Occupancy,0.798045
1,Previous_Day_Consumption,0.075982
2,Temperature_C,0.037304
3,Peak_Hour,0.033572
4,Day_of_Week,0.019164



Anomaly Summary:
Consumption_Status
Normal     8584
Unusual     176
Name: count, dtype: int64

✅ Backend successfully created.
✅ You can now run the website dashboard cell.


In [ ]:
# ============================================================
# 🚀 LAUNCH SMART ENERGY ADVISOR WEBSITE
# ============================================================

!pip -q install -U gradio

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_energy(
    hour,
    day_of_week,
    month,
    weekend,
    temperature,
    occupancy,
    peak_hour,
    previous_hour,
    previous_day
):

    input_data = pd.DataFrame({
        "Hour": [hour],
        "Day_of_Week": [day_of_week],
        "Month": [month],
        "Is_Weekend": [weekend],
        "Temperature_C": [temperature],
        "Occupancy": [occupancy],
        "Peak_Hour": [peak_hour],
        "Previous_Hour_Consumption": [previous_hour],
        "Previous_Day_Consumption": [previous_day]
    })

    # Prediction
    prediction = best_model.predict(input_data)[0]

    # Average consumption
    average = df["Energy_Consumption_kWh"].mean()

    # Difference
    difference = ((prediction - average) / average) * 100

    # Status
    if difference > 25:
        status = "🔴 HIGH"
    elif difference > 10:
        status = "🟠 MODERATE"
    else:
        status = "🟢 NORMAL"

    # Observations
    observations = []

    if temperature >= 30:
        observations.append(
            "High temperature may increase cooling-related energy demand."
        )

    if occupancy >= 120:
        observations.append(
            "High occupancy may increase lighting, HVAC and equipment demand."
        )

    if peak_hour == 1:
        observations.append(
            "This period is classified as a peak-demand period."
        )

    if prediction > previous_hour * 1.20:
        observations.append(
            "Predicted consumption is significantly higher than the previous hour."
        )

    if prediction > previous_day * 1.20:
        observations.append(
            "Predicted consumption is significantly higher than the previous day."
        )

    if not observations:
        observations.append(
            "No major energy-consumption risk factors were identified."
        )

    # Recommendations
    recommendations = []

    if temperature >= 30:
        recommendations.append(
            "Optimize air-conditioning and cooling settings."
        )

    if occupancy >= 120:
        recommendations.append(
            "Monitor lighting, fans, HVAC and electrical equipment."
        )

    if peak_hour == 1:
        recommendations.append(
            "Shift non-essential loads to off-peak hours where possible."
        )

    if prediction > previous_hour * 1.20:
        recommendations.append(
            "Check for unnecessary equipment or sudden load increases."
        )

    if prediction > previous_day * 1.20:
        recommendations.append(
            "Compare today's operating schedule with the previous day's usage."
        )

    recommendations.append(
        "Switch off unused electrical equipment and monitor avoidable consumption."
    )

    observations_text = "\n".join(
        "• " + x for x in observations
    )

    recommendations_text = "\n".join(
        "• " + x for x in recommendations
    )

    return f"""
# ⚡ Energy Analysis Result

## Predicted Energy Consumption

### `{prediction:.2f} kWh`

---

## 📊 Consumption Status

# {status}

**Historical Average:** `{average:.2f} kWh`

**Difference from Average:** `{difference:.2f}%`

---

## 🔍 Observations

{observations_text}

---

## 💡 Smart Energy Recommendations

{recommendations_text}
"""


# ============================================================
# GRAPH FUNCTIONS
# ============================================================

def trend_graph():

    fig = plt.figure(figsize=(10,4))

    plt.plot(
        df["DateTime"],
        df["Energy_Consumption_kWh"]
    )

    plt.xlabel("Time")
    plt.ylabel("Energy Consumption (kWh)")
    plt.title("Energy Consumption Trend")

    plt.tight_layout()

    return fig


def hourly_graph():

    hourly = df.groupby(
        "Hour"
    )["Energy_Consumption_kWh"].mean()

    fig = plt.figure(figsize=(10,4))

    plt.plot(
        hourly.index,
        hourly.values,
        marker="o"
    )

    plt.xlabel("Hour of Day")
    plt.ylabel("Average Energy Consumption (kWh)")
    plt.title("Average Energy Consumption by Hour")

    plt.xticks(range(24))

    plt.tight_layout()

    return fig


def feature_graph():

    fig = plt.figure(figsize=(9,5))

    plt.barh(
        feature_importance["Feature"],
        feature_importance["Importance"]
    )

    plt.xlabel("Importance Score")
    plt.ylabel("Feature")
    plt.title("Random Forest Feature Importance")

    plt.gca().invert_yaxis()

    plt.tight_layout()

    return fig


# ============================================================
# CREATE WEBSITE
# ============================================================

with gr.Blocks(
    title="Smart Energy Advisor"
) as app:

    # HEADER
    gr.Markdown("""
# ⚡ AI-Based Smart Energy Consumption Forecasting & Advisor

### Energy Forecasting • Analytics • Anomaly Detection • Smart Recommendations

**🌱 Primary SDG: SDG 7 – Affordable and Clean Energy**

---

Use this application to forecast energy consumption and receive
data-driven energy-management recommendations.
""")

    # ========================================================
    # TAB 1
    # ========================================================

    with gr.Tab("🔮 Energy Prediction"):

        gr.Markdown("""
## Enter Current Operating Conditions

Adjust the values below and click **ANALYZE ENERGY**.
""")

        with gr.Row():

            with gr.Column():

                hour = gr.Slider(
                    0, 23,
                    value=14,
                    step=1,
                    label="Hour of Day"
                )

                day = gr.Slider(
                    0, 6,
                    value=2,
                    step=1,
                    label="Day of Week (0 = Monday)"
                )

                month = gr.Slider(
                    1, 12,
                    value=7,
                    step=1,
                    label="Month"
                )

            with gr.Column():

                weekend = gr.Radio(
                    [0, 1],
                    value=0,
                    label="Weekend? (0 = No, 1 = Yes)"
                )

                temperature = gr.Slider(
                    10, 45,
                    value=32,
                    step=0.5,
                    label="Temperature (°C)"
                )

                occupancy = gr.Slider(
                    0, 300,
                    value=150,
                    step=1,
                    label="Occupancy"
                )

            with gr.Column():

                peak = gr.Radio(
                    [0, 1],
                    value=1,
                    label="Peak Hour? (0 = No, 1 = Yes)"
                )

                previous_hour = gr.Number(
                    value=45,
                    label="Previous Hour (kWh)"
                )

                previous_day = gr.Number(
                    value=42,
                    label="Previous Day (kWh)"
                )

        analyze_button = gr.Button(
            "⚡ ANALYZE ENERGY",
            variant="primary",
            size="lg"
        )

        result = gr.Markdown()

        analyze_button.click(
            fn=predict_energy,
            inputs=[
                hour,
                day,
                month,
                weekend,
                temperature,
                occupancy,
                peak,
                previous_hour,
                previous_day
            ],
            outputs=result
        )

    # ========================================================
    # TAB 2
    # ========================================================

    with gr.Tab("📊 Energy Analytics"):

        gr.Markdown(
            "## Energy Consumption Analytics"
        )

        gr.Plot(
            value=trend_graph()
        )

        gr.Plot(
            value=hourly_graph()
        )

    # ========================================================
    # TAB 3
    # ========================================================

    with gr.Tab("🤖 ML Performance"):

        gr.Markdown(f"""
## Machine Learning Model Comparison

### 🏆 Best Model: {best_model_name}

The models were evaluated using MAE, RMSE and R².
""")

        gr.Dataframe(
            value=results_df,
            interactive=False
        )

    # ========================================================
    # TAB 4
    # ========================================================

    with gr.Tab("🔍 Feature Importance"):

        gr.Markdown("""
## Energy Consumption Drivers

The Random Forest model identifies the variables that have
the greatest influence on energy-consumption prediction.
""")

        gr.Plot(
            value=feature_graph()
        )

        gr.Dataframe(
            value=feature_importance,
            interactive=False
        )

    # ========================================================
    # TAB 5
    # ========================================================

    with gr.Tab("🚨 Anomaly Detection"):

        gr.Markdown("""
## Unusual Energy Consumption

Isolation Forest is used to identify unusual consumption patterns.
""")

        anomaly_summary = (
            df["Consumption_Status"]
            .value_counts()
            .reset_index()
        )

        anomaly_summary.columns = [
            "Status",
            "Number of Records"
        ]

        gr.Dataframe(
            value=anomaly_summary,
            interactive=False
        )

    # ========================================================
    # TAB 6
    # ========================================================

    with gr.Tab("📁 Project Information"):

        gr.Markdown(f"""
# Project Overview

### AI-Based Smart Energy Consumption Forecasting & Advisor

**Primary SDG:** SDG 7 – Affordable and Clean Energy

**Dataset Records:** {len(df):,}

**Best Model:** {best_model_name}

### Main Components

✓ Energy consumption forecasting

✓ Machine learning model comparison

✓ Feature importance analysis

✓ Anomaly detection

✓ Smart Energy Advisor

✓ Energy-saving recommendations

### Dataset

The current demonstration uses simulated/synthetic energy data
for methodology and prototype development.

It should not be represented as actual measured electricity data.
""")

    # ========================================================
    # TAB 7
    # ========================================================

    with gr.Tab("🛡 Responsible AI"):

        gr.Markdown("""
# Responsible AI

### Fairness
The system does not use personal or sensitive attributes.

### Transparency
Model performance and feature importance are displayed.

### Privacy
No personally identifiable information is required.

### Human Oversight
The system provides recommendations and does not automatically
control electrical equipment.

### Limitations
The current prototype uses synthetic data.

Real-world deployment should be validated using actual
energy-meter data.

### Responsible Use
Predictions should support human decision-making rather than
replace professional energy-management decisions.
""")

    # ========================================================
    # FOOTER
    # ========================================================

    gr.Markdown("""
---

### 🌱 AI for Sustainability

**SDG 7 – Affordable and Clean Energy**

*AI for Sustainability Virtual Internship Project*
""")


# ============================================================
# 🚀 LAUNCH
# ============================================================

print("🚀 Starting Smart Energy Advisor...")
print("Please wait for the public URL...")

app.launch(
    share=True,
    debug=True
)

🚀 Starting Smart Energy Advisor...
Please wait for the public URL...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d34f7d347f790af357.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
